# 01 — Data Leakage Audit

**Purpose:** Systematically verify that no future information contaminates features or targets.

This is a critical validation step before any modeling.

Checks:
1. Automated leakage detection suite
2. Manual spot-check: lag alignment
3. Manual spot-check: target shift correctness
4. Temporal split integrity


In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from pathlib import Path
from rich import print

from src.validation.leakage_check import run_all_checks
from src.feature_engineering.builder import get_feature_columns

PROCESSED = Path('../../data/processed')

In [ ]:
df = pd.read_parquet(PROCESSED / 'enso_dataset.parquet').set_index('date')
feature_cols = get_feature_columns(df)

print(f'Dataset: {df.shape[0]} rows, {len(feature_cols)} feature columns')

## 1. Automated leakage check suite

In [ ]:
results = run_all_checks(df, feature_cols)
for check, passed in results.items():
    status = '[green]PASS[/green]' if passed else '[red]FAIL[/red]'
    print(f'{status}  {check}')

## 2. Manual spot-check: lag alignment

In [ ]:
# Pick 5 random rows and verify lag1 matches source value at t-1
sample_idx = df.index[10:20]  # avoid first rows where lags are NaN
check_rows = []
for ts in sample_idx[:5]:
    ts_lag1 = df.index[df.index.get_loc(ts) - 1]
    check_rows.append({
        'date':                  ts,
        'nino34_lag1_feature':   round(df.loc[ts, 'nino34_anom_lag1'], 4),
        'nino34_raw_at_t-1':     round(df.loc[ts_lag1, 'nino34_anom'], 4),
        'match':                 abs(df.loc[ts, 'nino34_anom_lag1'] - df.loc[ts_lag1, 'nino34_anom']) < 1e-6
    })
pd.DataFrame(check_rows)

## 3. Manual spot-check: target shift correctness

In [ ]:
# enso_t1 at row i must equal enso_phase at row i+1
check_target = []
for i in range(5, 10):
    ts   = df.index[i]
    ts_1 = df.index[i + 1]
    check_target.append({
        'date':         ts,
        'enso_t1':      df.loc[ts, 'enso_t1'],
        'enso_phase_t+1': df.loc[ts_1, 'enso_phase'],
        'match':        df.loc[ts, 'enso_t1'] == df.loc[ts_1, 'enso_phase']
    })
pd.DataFrame(check_target)